# 🧪 Experimentation Lead v4 — 9-arch 비교 (독립형)

| exp | 아키텍처 | 비고 |
|-----|---------|------|
| exp_1 | FasterRCNN-R50 | torchvision |
| exp_2 | FasterRCNN-R34 | torchvision |
| exp_3 | RetinaNet | torchvision |
| exp_4 | Swin-V2-T+FPN | torchvision |
| exp_5 | RT-DETR-L | ultralytics |
| exp_6 | YOLOv5s | ultralytics |
| exp_7 | RF-DETR Small | rfdetr |
| exp_8 | RF-DETR Medium | rfdetr |
| exp_9 | YOLOv8m | ultralytics |

> 모든 실험은 **fold0** 기준, Google Drive 데이터 사용


## 0. GPU / 패키지 점검


In [ ]:
import torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    total = torch.cuda.get_device_properties(0).total_memory/1e9
    print(f'GPU : {torch.cuda.get_device_name(0)} ({total:.1f}GB)')
else:
    print('⚠️  CPU 모드')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
!pip install -q 'rfdetr[train,loggers]' ultralytics easyocr torchmetrics
print('✅ 설치 완료')


## 1. Drive 마운트 + 데이터 준비 (fold0)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os, json, glob, shutil, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from collections import defaultdict
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ── 경로 탐색 ─────────────────────────────────────────────
SHARED_ID = '1SlNpnNKD7cPbwLJH0A9-WUcx1Fhbzfrd'
CANDS = [
    f'/content/drive/.shortcut-targets-by-id/{SHARED_ID}/ai12-level1-project/sprint_ai_project1_data',
    '/content/drive/MyDrive/1팀 공유 문서/ai12-level1-project/sprint_ai_project1_data',
    '/content/drive/MyDrive/ai12-level1-project/sprint_ai_project1_data',
]
DATA_ROOT = next((c for c in CANDS if os.path.exists(c)), None)
assert DATA_ROOT, 'sprint_ai_project1_data 못 찾음'
PROJ_ROOT = os.path.dirname(DATA_ROOT)
print(f'DATA_ROOT: {DATA_ROOT}')

# ── dataset_74_5fold.zip 복원 ──────────────────────────────
DATASET_DIR = '/content/dataset_74'
ZIP_CANDS = [
    os.path.join(PROJ_ROOT, 'dataset_74_5fold.zip'),
    f'/content/drive/.shortcut-targets-by-id/{SHARED_ID}/ai12-level1-project/dataset_74_5fold.zip',
]
ZIP = next((z for z in ZIP_CANDS if os.path.exists(z)), None)
assert ZIP, 'dataset_74_5fold.zip 못 찾음'

if not os.path.exists(DATASET_DIR) or not os.listdir(DATASET_DIR):
    print('zip 복원 중...')
    shutil.copy(ZIP, '/content/dataset_74_5fold.zip')
    os.system('unzip -qo /content/dataset_74_5fold.zip -d /content/dataset_74')
    os.remove('/content/dataset_74_5fold.zip')

FOLD0_TRAIN_DIR = os.path.join(DATASET_DIR,'fold0','train')
FOLD0_VAL_DIR   = os.path.join(DATASET_DIR,'fold0','valid')
FOLD0_TRAIN_ANN = os.path.join(FOLD0_TRAIN_DIR,'_annotations.coco.json')
FOLD0_VAL_ANN   = os.path.join(FOLD0_VAL_DIR,  '_annotations.coco.json')

with open(FOLD0_TRAIN_ANN) as f: _tr = json.load(f)
with open(FOLD0_VAL_ANN)   as f: _va = json.load(f)
cats = sorted([c for c in _tr['categories'] if c['id']!=0], key=lambda x:x['id'])
NUM_CLASSES  = len(cats)
CAT_ID_TO_LABEL = {c['id']: i+1 for i,c in enumerate(cats)}  # 1-indexed (0=bg)
LABEL_TO_NAME   = {i+1: str(c['name']) for i,c in enumerate(cats)}

DEFAULT_CONFIG = {
    'batch_size': 4,
    'img_size': 512,
    'freeze_epochs': 3,
    'anchor_sizes': ((32,),(64,),(128,),(256,),(512,)),
    'max_detections_per_img': 100,
    'submission_top_k': 4,
}
BATCH_SIZE = DEFAULT_CONFIG['batch_size']
IMG_SIZE   = DEFAULT_CONFIG['img_size']
print(f'클래스 수  : {NUM_CLASSES}')
print(f'train      : {len(_tr["images"])}장')
print(f'valid      : {len(_va["images"])}장')


## 1-1. Copy-Paste Balanced 증강 적용


In [ ]:
# ── copy_paste_balanced 증강 인라인 구현 ─────────────────────
import random, copy, math
import numpy as np
from PIL import Image
from collections import defaultdict

def _get_freq(annotations):
    freq = defaultdict(int)
    for ann in annotations:
        freq[ann['category_id']] += 1
    return freq

def _inv_freq_weight(cat_id, freq, total_imgs):
    cnt = freq.get(cat_id, 1)
    return total_imgs / max(cnt, 1)

def _paste_pill(dst_img, dst_anns, src_img, src_ann, dst_w, dst_h):
    """src 이미지에서 알약 크롭을 dst에 붙여넣기"""
    x,y,w,h = [int(v) for v in src_ann['bbox']]
    if w<=0 or h<=0: return dst_anns
    margin = 5
    x1=max(0,x-margin); y1=max(0,y-margin)
    x2=min(src_img.width,x+w+margin); y2=min(src_img.height,y+h+margin)
    crop = src_img.crop((x1,y1,x2,y2))
    # 랜덤 크기 조정
    scale = random.uniform(0.7, 1.2)
    nw = max(10, int(crop.width*scale)); nh = max(10, int(crop.height*scale))
    crop = crop.resize((nw,nh), Image.BILINEAR)
    # 랜덤 위치 붙여넣기
    px = random.randint(0, max(0, dst_w-nw))
    py = random.randint(0, max(0, dst_h-nh))
    # 알파 마스크
    if crop.mode != 'RGBA':
        mask = Image.new('L', crop.size, 220)
    else:
        mask = crop.split()[3]
        crop = crop.convert('RGB')
    dst_img.paste(crop, (px,py), mask)
    new_ann = copy.deepcopy(src_ann)
    new_ann['bbox'] = [float(px),float(py),float(nw),float(nh)]
    new_ann['area'] = float(nw*nh)
    dst_anns.append(new_ann)
    return dst_anns


def augment_fold(strategy, fold_idx, dataset_dir, out_dir, aug_factor=2):
    """copy_paste_balanced 증강"""
    import shutil, os, json
    assert strategy=='copy_paste_balanced', f'지원하지 않는 전략: {strategy}'

    src_train = os.path.join(dataset_dir,f'fold{fold_idx}','train')
    src_val   = os.path.join(dataset_dir,f'fold{fold_idx}','valid')
    dst_train = os.path.join(out_dir,    f'fold{fold_idx}','train')
    dst_val   = os.path.join(out_dir,    f'fold{fold_idx}','valid')
    os.makedirs(dst_train,exist_ok=True)
    os.makedirs(dst_val,  exist_ok=True)

    # val은 그대로 복사
    with open(os.path.join(src_val,'_annotations.coco.json')) as f: val_coco=json.load(f)
    for img in val_coco['images']:
        src_p=os.path.join(src_val,img['file_name']); dst_p=os.path.join(dst_val,img['file_name'])
        if os.path.exists(src_p) and not os.path.exists(dst_p): shutil.copy(src_p,dst_p)
    with open(os.path.join(dst_val,'_annotations.coco.json'),'w') as f: json.dump(val_coco,f)

    # train 증강
    with open(os.path.join(src_train,'_annotations.coco.json')) as f: tr=json.load(f)
    freq = _get_freq(tr['annotations'])
    id2img  = {img['id']:img for img in tr['images']}
    ann_by  = defaultdict(list)
    for ann in tr['annotations']: ann_by[ann['image_id']].append(ann)

    new_imgs=[]; new_anns=[]
    # 기존 이미지 복사
    for img in tr['images']:
        src_p=os.path.join(src_train,img['file_name']); dst_p=os.path.join(dst_train,img['file_name'])
        if os.path.exists(src_p) and not os.path.exists(dst_p): shutil.copy(src_p,dst_p)
        new_imgs.append(img)
        for ann in ann_by[img['id']]: new_anns.append(ann)

    # 증강 이미지 생성
    n_orig   = len(tr['images'])
    n_gen    = n_orig*(aug_factor-1)
    max_iid  = max(img['id'] for img in tr['images'])
    max_aid  = max((ann['id'] for ann in tr['annotations']),default=0)

    # 클래스별 이미지 인덱스 구축 (희귀 클래스 우선 샘플링용)
    cat_to_imgs=defaultdict(list)
    for img in tr['images']:
        for ann in ann_by[img['id']]: cat_to_imgs[ann['category_id']].append(img['id'])

    total_imgs=len(tr['images'])
    cats=[ann['category_id'] for ann in tr['annotations']]
    weights=[_inv_freq_weight(c,freq,total_imgs) for c in cats]
    w_sum=sum(weights)
    weights=[w/w_sum for w in weights]

    from tqdm.auto import tqdm
    for gi in tqdm(range(n_gen),desc='  Augment'):
        # 배경 이미지 (랜덤)
        bg_img_info = random.choice(tr['images'])
        bg_path     = os.path.join(src_train,bg_img_info['file_name'])
        if not os.path.exists(bg_path): continue
        bg_pil  = Image.open(bg_path).convert('RGB')
        bg_anns = [copy.deepcopy(a) for a in ann_by[bg_img_info['id']]]

        # 역빈도 가중치로 소스 어노테이션 선택
        n_paste = random.randint(1,3)
        chosen  = random.choices(range(len(tr['annotations'])),weights=weights,k=n_paste)
        for ci in chosen:
            src_ann  = tr['annotations'][ci]
            src_info = id2img.get(src_ann['image_id'])
            if src_info is None: continue
            src_path = os.path.join(src_train,src_info['file_name'])
            if not os.path.exists(src_path): continue
            src_pil  = Image.open(src_path).convert('RGB')
            bg_anns  = _paste_pill(bg_pil,bg_anns,src_pil,src_ann,
                                   bg_pil.width,bg_pil.height)

        # 저장
        new_fname = f'aug_{gi:05d}.png'
        new_iid   = max_iid+gi+1
        bg_pil.save(os.path.join(dst_train,new_fname))
        new_imgs.append({'id':new_iid,'file_name':new_fname,
                         'width':bg_pil.width,'height':bg_pil.height})
        for ann in bg_anns:
            max_aid+=1
            new_anns.append({**ann,'id':max_aid,'image_id':new_iid})

    aug_coco={'images':new_imgs,'annotations':new_anns,'categories':tr['categories']}
    with open(os.path.join(dst_train,'_annotations.coco.json'),'w') as f:
        json.dump(aug_coco,f)
    print(f'  원본 {n_orig}장 + 증강 {n_gen}장 = 총 {len(new_imgs)}장')

print('✅ copy_paste_balanced 증강 함수 정의 완료')


In [ ]:
# 증강된 경로로 DataLoader 재생성
train_ds = COCOPillDataset(FOLD0_TRAIN_DIR, FOLD0_TRAIN_ANN, IMG_SIZE)
val_ds   = COCOPillDataset(FOLD0_VAL_DIR,   FOLD0_VAL_ANN,   IMG_SIZE)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, collate_fn=collate_fn)
print(f'train_loader: {len(train_loader)}배치 ({len(train_ds)}장)')
print(f'val_loader  : {len(val_loader)}배치 ({len(val_ds)}장)')


## 2. Dataset + DataLoader 구현


In [ ]:
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class COCOPillDataset(Dataset):
    def __init__(self, img_dir, ann_path, img_size=512):
        self.img_dir  = img_dir
        self.img_size = img_size
        with open(ann_path) as f: coco = json.load(f)
        self.imgs = coco['images']
        self.ann_by = defaultdict(list)
        for ann in coco['annotations']:
            if ann.get('iscrowd',0)==0 and ann['category_id']>0:
                self.ann_by[ann['image_id']].append(ann)
        self.tfm = T.Compose([
            T.Resize((img_size, img_size)),
            T.ToTensor(),
            T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])
    def __len__(self): return len(self.imgs)
    def __getitem__(self, idx):
        info  = self.imgs[idx]
        fname = info['file_name']
        path  = os.path.join(self.img_dir, fname)
        img   = Image.open(path).convert('RGB')
        W,H   = img.size
        sx,sy = self.img_size/W, self.img_size/H
        img   = self.tfm(img)
        anns  = self.ann_by.get(info['id'],[])
        boxes,labels=[],[]
        for a in anns:
            x,y,w,h=a['bbox']
            if w>0 and h>0:
                boxes.append([x*sx,y*sy,(x+w)*sx,(y+h)*sy])
                labels.append(CAT_ID_TO_LABEL.get(a['category_id'],1))
        target = {
            'boxes' :torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0,4)),
            'labels':torch.tensor(labels,dtype=torch.int64)   if labels else torch.zeros(0,dtype=torch.int64),
            'image_id':torch.tensor([info['id']]),
        }
        return img, target

def collate_fn(batch): return tuple(zip(*batch))

train_ds = COCOPillDataset(FOLD0_TRAIN_DIR, FOLD0_TRAIN_ANN, IMG_SIZE)
val_ds   = COCOPillDataset(FOLD0_VAL_DIR,   FOLD0_VAL_ANN,   IMG_SIZE)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=0, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, collate_fn=collate_fn)
print(f'train_loader: {len(train_loader)}배치')
print(f'val_loader  : {len(val_loader)}배치')


## 3. 모델 팩토리


In [ ]:
import torchvision
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn_v2, retinanet_resnet50_fpn_v2,
    FasterRCNN, RetinaNet,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.retinanet import RetinaNetClassificationHead
from torchvision.models import resnet34, ResNet34_Weights, swin_v2_t, Swin_V2_T_Weights
from torchvision.models.detection.backbone_utils import BackboneWithFPN   # ← 최상위 import
from torchvision.ops import misc as misc_nn_ops
from functools import partial

def get_model(arch, num_classes, anchor_sizes=None, max_det=100):
    nc = num_classes + 1

    if arch == 'fasterrcnn':
        m = fasterrcnn_resnet50_fpn_v2(weights='DEFAULT')
        in_f = m.roi_heads.box_predictor.cls_score.in_features
        m.roi_heads.box_predictor = FastRCNNPredictor(in_f, nc)
        m.roi_heads.detections_per_img = max_det

    elif arch == 'fasterrcnn_r34':
        # ── fix: BackboneWithFPN 최상위에서 import, 함수 내 재import 제거 ──
        backbone_body = resnet34(weights=ResNet34_Weights.DEFAULT)
        return_layers = {'layer1':'0','layer2':'1','layer3':'2','layer4':'3'}
        in_channels_list = [64,128,256,512]
        bfpn = BackboneWithFPN(backbone_body, return_layers, in_channels_list,
                               out_channels=256,
                               extra_blocks=misc_nn_ops.LastLevelMaxPool())
        m = FasterRCNN(bfpn, num_classes=nc)
        m.roi_heads.detections_per_img = max_det

    elif arch == 'retinanet':
        m = retinanet_resnet50_fpn_v2(weights='DEFAULT')
        num_anchors = m.head.classification_head.num_anchors
        m.head.classification_head = RetinaNetClassificationHead(
            in_channels=256, num_anchors=num_anchors, num_classes=num_classes,
            norm_layer=partial(nn.GroupNorm,32))

    elif arch == 'swin':
        # ── fix: Swin 출력 [B,H,W,C] → permute → [B,C,H,W] + proj 256ch ──
        swin = swin_v2_t(weights=Swin_V2_T_Weights.DEFAULT)
        class SwinBackbone(nn.Module):
            def __init__(self, swin_model):
                super().__init__()
                self.features = swin_model.features
                self.norm     = swin_model.norm
                self.proj     = nn.Conv2d(768, 256, 1)
                self.out_channels = 256
            def forward(self, x):
                x = self.features(x)          # [B,H,W,768]
                x = self.norm(x)
                x = x.permute(0,3,1,2)        # [B,768,H,W]
                x = self.proj(x)              # [B,256,H,W]
                return {'0': x}
        sb = SwinBackbone(swin)
        m  = FasterRCNN(sb, num_classes=nc)
        m.roi_heads.detections_per_img = max_det

    else:
        raise ValueError(f'unknown arch: {arch}')
    return m


def freeze_backbone(model, freeze=True):
    for name, param in model.named_parameters():
        if 'backbone' in name or 'body' in name or 'features' in name:
            param.requires_grad = not freeze

def count_parameters(model):
    total      = sum(p.numel() for p in model.parameters())
    trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {'total':total,'trainable':trainable}

def keep_top_k_per_image(preds, k=4):
    out=[]
    for p in preds:
        if len(p['scores'])==0: out.append(p); continue
        idx=p['scores'].topk(min(k,len(p['scores']))).indices
        out.append({kk:vv[idx] for kk,vv in p.items() if kk!='image_id'})
    return out

print('✅ 모델 팩토리 정의 완료')


## 4. 학습·평가 함수


In [ ]:
from torchmetrics.detection import MeanAveragePrecision

def build_optimizer_scheduler(model, cfg, freeze_epochs=3):
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg['lr'], weight_decay=1e-4)
    if cfg['scheduler']=='cosine':
        sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=cfg['epochs'],eta_min=1e-6)
    else:
        sch = torch.optim.lr_scheduler.ConstantLR(opt,factor=1.0)
    return opt, sch


def train_one_epoch(model, loader, optimizer, scaler):
    model.train(); running,n=0.0,0
    for images,targets in tqdm(loader,desc='  Train',leave=False):
        images  = [img.to(device) for img in images]
        targets = [{k:v.to(device) for k,v in t.items()} for t in targets]
        optimizer.zero_grad(set_to_none=True)
        try:
            with torch.autocast(device_type=device.type,dtype=torch.float16,
                                enabled=(device.type=='cuda')):
                loss_dict = model(images, targets)
                loss = sum(loss_dict.values())
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(),5.0)
            scaler.step(optimizer); scaler.update()
            running+=loss.item(); n+=1
        except torch.cuda.OutOfMemoryError:
            print('  OOM — 스킵'); optimizer.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()
    return running/max(n,1)


@torch.no_grad()
def evaluate_map(model, loader, score_thr=0.05):
    model.eval()
    metric = MeanAveragePrecision(iou_type='bbox',class_metrics=False)
    for images,targets in tqdm(loader,desc='  Eval',leave=False):
        images=[img.to(device) for img in images]
        preds=model(images)
        filt=[]
        for p in preds:
            keep=p['scores'].cpu()>=score_thr
            filt.append({'boxes':p['boxes'].cpu()[keep],
                         'scores':p['scores'].cpu()[keep],
                         'labels':p['labels'].cpu()[keep]})
        metric.update(filt,[{k:v.cpu() for k,v in t.items()} for t in targets])
    return metric.compute()


def run_torchvision_experiment(cfg, num_classes):
    print(f"\n{'='*60}\n> {cfg['exp_id']} | {cfg['arch']}\n{'='*60}")
    model = get_model(cfg['arch'], num_classes,
                      max_det=DEFAULT_CONFIG['max_detections_per_img']).to(device)
    freeze_backbone(model, freeze=True)
    opt,sch=build_optimizer_scheduler(model,cfg)
    scaler = (torch.amp.GradScaler(device.type)
              if hasattr(torch.amp,'GradScaler')
              else torch.cuda.amp.GradScaler())
    losses=[]; t0=time.time()
    for epoch in range(1,cfg['epochs']+1):
        if epoch==DEFAULT_CONFIG['freeze_epochs']+1:
            freeze_backbone(model,freeze=False)
            opt,sch=build_optimizer_scheduler(model,cfg)
        avg=train_one_epoch(model,train_loader,opt,scaler)
        sch.step(); losses.append(avg)
        print(f'  Epoch {epoch}/{cfg["epochs"]} | Loss={avg:.4f}')
    elapsed=time.time()-t0
    val_res=evaluate_map(model,val_loader)
    p=count_parameters(model)
    result={'exp_id':cfg['exp_id'],'arch':cfg['arch'],'lr':cfg['lr'],
            'scheduler':cfg['scheduler'],'epochs':cfg['epochs'],
            'params_M':round(p['total']/1e6,1),'train_loss':round(losses[-1],4),
            'mAP':val_res['map'].item(),'mAP_50':val_res['map_50'].item(),
            'mAP_75':val_res['map_75'].item(),'elapsed_sec':round(elapsed,1),
            'note':cfg.get('note','')}
    print(f'  Done | mAP={result["mAP"]:.4f} | {elapsed:.0f}s')
    del opt,sch; torch.cuda.empty_cache()
    return model, losses, val_res, result

print('✅ 학습/평가 함수 정의 완료')


## 5. YOLO 형식 변환 + ultralytics 실험 함수


In [ ]:
YOLO_DIR = '/content/yolo_fold0'

def build_yolo_dataset(coco_dir, ann_path, yolo_split_dir):
    img_dir = os.path.join(yolo_split_dir,'images')
    lbl_dir = os.path.join(yolo_split_dir,'labels')
    os.makedirs(img_dir,exist_ok=True); os.makedirs(lbl_dir,exist_ok=True)
    with open(ann_path) as f: coco=json.load(f)
    cats=[c for c in coco['categories'] if c['id']!=0]
    cat2yolo={c['id']:i for i,c in enumerate(sorted(cats,key=lambda x:x['id']))}
    id2img={img['id']:img for img in coco['images']}
    ann_by=defaultdict(list)
    for ann in coco['annotations']:
        if ann['category_id']!=0: ann_by[ann['image_id']].append(ann)
    for img_info in coco['images']:
        fname=img_info['file_name']
        W,H=img_info['width'],img_info['height']
        src=os.path.join(coco_dir,fname); dst=os.path.join(img_dir,fname)
        if os.path.exists(src) and not os.path.exists(dst): shutil.copy(src,dst)
        lbl=os.path.join(lbl_dir,os.path.splitext(fname)[0]+'.txt')
        anns=ann_by[img_info['id']]
        if not anns: open(lbl,'w').close(); continue
        with open(lbl,'w') as f:
            for a in anns:
                yc=cat2yolo.get(a['category_id']); 
                if yc is None: continue
                x,y,w,h=a['bbox']
                xc=(x+w/2)/W; yc_n=(y+h/2)/H; wn=w/W; hn=h/H
                xc=max(0,min(1,xc)); yc_n=max(0,min(1,yc_n))
                wn=max(0,min(1,wn)); hn=max(0,min(1,hn))
                f.write(f'{yc} {xc:.6f} {yc_n:.6f} {wn:.6f} {hn:.6f}\n')
    return cats

# YOLO 변환
print('YOLO 형식 변환 중...')
cats_yolo = build_yolo_dataset(FOLD0_TRAIN_DIR,FOLD0_TRAIN_ANN,os.path.join(YOLO_DIR,'train'))
build_yolo_dataset(FOLD0_VAL_DIR,FOLD0_VAL_ANN,os.path.join(YOLO_DIR,'valid'))
names=[str(c['name']) for c in sorted(cats_yolo,key=lambda x:x['id'])]
yaml_path=os.path.join(YOLO_DIR,'data.yaml')
with open(yaml_path,'w') as f:
    f.write(f'path: {YOLO_DIR}\ntrain: train/images\nval: valid/images\n')
    f.write(f'nc: {len(cats_yolo)}\nnames: {names}\n')
print(f'✅ YOLO 변환 완료: {yaml_path}')


In [ ]:
from ultralytics import YOLO as UltralyticsYOLO
from rfdetr import RFDETRSmall, RFDETRMedium

def run_ultralytics_experiment(cfg):
    arch=cfg['arch']
    print(f"\n{'='*60}\n> {cfg['exp_id']} | {arch}\n{'='*60}")
    if arch=='yolov5':   ckpt='yolov5s.pt'
    elif arch=='rtdetr': ckpt=cfg.get('variant','rtdetr-l.pt')
    elif arch=='yolov8m':ckpt='yolov8m.pt'
    else: raise ValueError(arch)
    model_ul=UltralyticsYOLO(ckpt)
    dev='0' if torch.cuda.is_available() else 'cpu'
    t0=time.time()
    kw=dict(data=yaml_path,epochs=cfg['epochs'],lr0=cfg['lr'],imgsz=IMG_SIZE,
            batch=BATCH_SIZE,device=dev,patience=5,
            project=f'runs/{arch}',name=cfg['exp_id'],exist_ok=True,verbose=False)
    if arch=='yolov8m': kw.update(degrees=180.0,fliplr=0.5,mosaic=1.0)
    model_ul.train(**kw)
    elapsed=time.time()-t0
    vr=model_ul.val(data=yaml_path,verbose=False)
    mAP=float(getattr(vr.box,'map',0) or 0)
    mAP50=float(getattr(vr.box,'map50',0) or 0)
    mAP75=float(getattr(vr.box,'map75',0) or 0)
    n_params=round(sum(p.numel() for p in model_ul.model.parameters())/1e6,1)
    result={'exp_id':cfg['exp_id'],'arch':arch,'lr':cfg['lr'],
            'scheduler':cfg['scheduler'],'epochs':cfg['epochs'],
            'params_M':n_params,'train_loss':0.0,
            'mAP':mAP,'mAP_50':mAP50,'mAP_75':mAP75,
            'elapsed_sec':round(elapsed,1),'note':cfg.get('note','')}
    print(f'  Done | mAP={mAP:.4f} | {elapsed:.0f}s')
    torch.cuda.empty_cache()
    return model_ul,[],vr,result


def run_rfdetr_experiment(cfg):
    arch=cfg['arch']
    print(f"\n{'='*60}\n> {cfg['exp_id']} | {arch}\n{'='*60}")
    ModelCls=RFDETRSmall if arch=='rfdetr_small' else RFDETRMedium
    model_rf=ModelCls()
    out_dir=f'/tmp/rfdetr_{arch}'; os.makedirs(out_dir,exist_ok=True)
    t0=time.time()
    model_rf.train(dataset_dir=DATASET_DIR+'/fold0',output_dir=out_dir,
        epochs=cfg['epochs'],batch_size=BATCH_SIZE,
        lr=cfg['lr'],lr_encoder=cfg['lr']*1.5,weight_decay=1e-4,
        lr_scheduler='cosine',warmup_epochs=0,
        early_stopping=True,early_stopping_patience=10,
        early_stopping_min_delta=0.001,tensorboard=False)
    elapsed=time.time()-t0
    from torchmetrics.detection import MeanAveragePrecision
    metric=MeanAveragePrecision(iou_type='bbox')
    with open(FOLD0_VAL_ANN) as f: cv=json.load(f)
    ab={}
    for a in cv['annotations']: ab.setdefault(a['image_id'],[]).append(a)
    for img in cv['images']:
        fp=os.path.join(FOLD0_VAL_DIR,img['file_name'])
        if not os.path.exists(fp): continue
        from PIL import Image as PILImage
        dets=model_rf.predict(PILImage.open(fp).convert('RGB'),threshold=0.05)
        gt=ab.get(img['id'],[])
        gb=[[a['bbox'][0],a['bbox'][1],a['bbox'][0]+a['bbox'][2],a['bbox'][1]+a['bbox'][3]] for a in gt]
        gl=[CAT_ID_TO_LABEL.get(a['category_id'],1) for a in gt]
        metric.update(
            [{'boxes':torch.tensor(dets.xyxy,dtype=torch.float32) if len(dets.xyxy) else torch.zeros((0,4)),
              'scores':torch.tensor(dets.confidence,dtype=torch.float32) if len(dets.confidence) else torch.zeros(0),
              'labels':torch.tensor(dets.class_id,dtype=torch.int64) if len(dets.class_id) else torch.zeros(0,dtype=torch.int64)}],
            [{'boxes':torch.tensor(gb,dtype=torch.float32) if gb else torch.zeros((0,4)),
              'labels':torch.tensor(gl,dtype=torch.int64) if gl else torch.zeros(0,dtype=torch.int64)}])
    res=metric.compute()
    mAP=res['map'].item(); mAP50=res['map_50'].item(); mAP75=res['map_75'].item()
    try:
        n_params=round(sum(p.numel() for p in model_rf.model.parameters())/1e6,1)
    except:
        n_params=36.0 if arch=='rfdetr_small' else 74.0  # 근사값
    result={'exp_id':cfg['exp_id'],'arch':arch,'lr':cfg['lr'],
            'scheduler':cfg['scheduler'],'epochs':cfg['epochs'],
            'params_M':n_params,'train_loss':0.0,
            'mAP':mAP,'mAP_50':mAP50,'mAP_75':mAP75,
            'elapsed_sec':round(elapsed,1),'note':cfg.get('note','')}
    print(f'  Done | mAP={mAP:.4f} | {elapsed:.0f}s')
    torch.cuda.empty_cache()
    return model_rf,[],res,result

print('✅ ultralytics / RF-DETR 실험 함수 정의 완료')


## 6. 실험 설계 (9개)


In [ ]:
experiment_configs = [
    {'exp_id':'exp_1','arch':'fasterrcnn',    'lr':1e-4,'scheduler':'cosine','epochs':5,'note':'R50 baseline'},
    {'exp_id':'exp_2','arch':'fasterrcnn_r34','lr':1e-4,'scheduler':'cosine','epochs':5,'note':'R34 경량'},
    {'exp_id':'exp_3','arch':'retinanet',     'lr':1e-4,'scheduler':'cosine','epochs':5,'note':'1-stage'},
    {'exp_id':'exp_4','arch':'swin',          'lr':5e-5,'scheduler':'cosine','epochs':5,'note':'Swin V2-T'},
    {'exp_id':'exp_5','arch':'rtdetr',        'lr':1e-4,'scheduler':'cosine','epochs':5,'note':'RT-DETR-L','variant':'rtdetr-l.pt'},
    {'exp_id':'exp_6','arch':'yolov5',        'lr':1e-3,'scheduler':'cosine','epochs':5,'note':'YOLOv5s'},
    {'exp_id':'exp_7','arch':'rfdetr_small',  'lr':1e-4,'scheduler':'cosine','epochs':5,'note':'RF-DETR Small'},
    {'exp_id':'exp_8','arch':'rfdetr_medium', 'lr':1e-4,'scheduler':'cosine','epochs':5,'note':'RF-DETR Medium'},
    {'exp_id':'exp_9','arch':'yolov8m',       'lr':1e-3,'scheduler':'cosine','epochs':5,'note':'YOLOv8m'},
]

import pandas as pd
display(pd.DataFrame(experiment_configs))


## 7. 실험 루프 실행

> 특정 실험만 하려면 슬라이싱: `experiment_configs[6:]`


In [ ]:
all_results, all_models, all_histories = [], {}, {}
TORCHVISION_ARCHS   = {'fasterrcnn','fasterrcnn_r34','retinanet','swin'}
ULTRALYTICS_ARCHS   = {'yolov5','rtdetr','yolov8m'}
RFDETR_ARCHS        = {'rfdetr_small','rfdetr_medium'}

for cfg in experiment_configs:
    try:
        if cfg['arch'] in TORCHVISION_ARCHS:
            model,history,val_res,result=run_torchvision_experiment(cfg,NUM_CLASSES)
        elif cfg['arch'] in ULTRALYTICS_ARCHS:
            model,history,val_res,result=run_ultralytics_experiment(cfg)
        elif cfg['arch'] in RFDETR_ARCHS:
            model,history,val_res,result=run_rfdetr_experiment(cfg)
        else:
            raise ValueError(f'unknown arch: {cfg["arch"]}')
        all_results.append(result)
        all_models[cfg['exp_id']]   =model
        all_histories[cfg['exp_id']]=history
    except Exception as e:
        print(f'  {cfg["exp_id"]} 실패: {e}')
        all_results.append({'exp_id':cfg['exp_id'],'arch':cfg['arch'],
                             'mAP':0,'mAP_50':0,'mAP_75':0,'note':str(e)[:80]})

print('\n✅ 전체 실험 완료')


## 8. 성능 비교표 + 시각화


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm, subprocess
subprocess.run(['apt-get','install','-y','-q','fonts-nanum'],capture_output=True)
fm.fontManager.__init__()
plt.rcParams['font.family']='NanumGothic'
plt.rcParams['axes.unicode_minus']=False

results_df = pd.DataFrame(all_results).sort_values('mAP',ascending=False).reset_index(drop=True)
display(
    results_df.style
    .highlight_max(subset=[c for c in ['mAP','mAP_50','mAP_75'] if c in results_df.columns],color='#d4edda')
    .highlight_min(subset=[c for c in ['params_M','elapsed_sec'] if c in results_df.columns],color='#cce5ff')
    .set_caption('실험 결과 비교 (초록=최고 / 파랑=최소)')
    .format({'mAP':'{:.4f}','mAP_50':'{:.4f}','mAP_75':'{:.4f}','params_M':'{:.1f}M','elapsed_sec':'{:.0f}s'},na_rep='—')
)
best_exp_id = results_df.iloc[0]['exp_id']
print(f'\n최고 성능: {best_exp_id}  (mAP={results_df.iloc[0]["mAP"]:.4f})')


In [ ]:
fig, axes = plt.subplots(1,3,figsize=(17,5))

# 1) 학습 손실 곡선
for exp_id,hist in all_histories.items():
    if hist: axes[0].plot(hist,label=exp_id)
axes[0].set_title('Train Loss 곡선'); axes[0].legend(fontsize=7)
axes[0].set_xlabel('Epoch'); axes[0].grid(True,alpha=0.3)

# 2) mAP 막대
df_s=results_df.dropna(subset=['mAP'])
colors=['#2dc653' if e==best_exp_id else '#457b9d' for e in df_s['exp_id']]
axes[1].bar(df_s['exp_id'],df_s['mAP'],color=colors,alpha=0.85)
axes[1].set_title('mAP@0.5:0.95'); axes[1].set_xticklabels(df_s['exp_id'],rotation=45,ha='right')
axes[1].axhline(df_s['mAP'].max(),color='red',linestyle='--',linewidth=1)
axes[1].grid(True,alpha=0.3,axis='y')

# 3) 파라미터 vs mAP 산점도
for _,row in df_s.iterrows():
    if pd.notna(row.get('params_M')) and pd.notna(row.get('mAP')):
        axes[2].scatter(row['params_M'],row['mAP'],s=80)
        axes[2].annotate(row['exp_id'],(row['params_M'],row['mAP']),
                         fontsize=7,xytext=(3,3),textcoords='offset points')
axes[2].set_title('파라미터 수 vs mAP')
axes[2].set_xlabel('파라미터 (M)'); axes[2].set_ylabel('mAP')
axes[2].grid(True,alpha=0.3)

plt.suptitle('9-arch 실험 비교 (fold0)',fontsize=13)
plt.tight_layout()
plt.savefig('/content/experiment_comparison.png',dpi=100,bbox_inches='tight')
plt.show()


## 9. 결과 저장


In [ ]:
import datetime

# Drive에 결과 저장
MY_OUTPUT = '/content/drive/MyDrive/pill_detection_outputs'
os.makedirs(MY_OUTPUT, exist_ok=True)

results_df.to_csv(os.path.join(MY_OUTPUT,'experiment_v4_results.csv'), index=False)
shutil.copy('/content/experiment_comparison.png',
            os.path.join(MY_OUTPUT,'experiment_v4_comparison.png'))

best_model = all_models.get(best_exp_id)
if best_model is not None and hasattr(best_model,'state_dict'):
    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M')
    ckpt_path = os.path.join(MY_OUTPUT,f'best_{best_exp_id}_{ts}.pth')
    torch.save(best_model.state_dict(), ckpt_path)
    print(f'✅ best model 저장: {ckpt_path}')

print(f'결과 CSV: {MY_OUTPUT}/experiment_v4_results.csv')
print(f'비교 그래프: {MY_OUTPUT}/experiment_v4_comparison.png')


## 결과 요약

| 항목 | 내용 |
|------|------|
| 총 실험 수 | 9개 |
| torchvision | FasterRCNN-R50 / R34 / RetinaNet / Swin-V2-T |
| ultralytics | YOLOv5s / RT-DETR-L / YOLOv8m |
| RF-DETR | Small / Medium |
| 데이터 | dataset_74_5fold.zip — fold0 |
| 외부 의존성 | data_pipeline.py / model_factory.py 없음 |
